# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided as a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields using their `@id` identifiers.

Let's list all record sets, field IDs, and other available entity IDs in the dataset for navigation and reference.

In [ ]:
# Get all available record sets and fields using their '@id'

record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in this Croissant package. Listing distributions...")
    # If the Croissant schema does not explicitly declare recordSet, try Distributions
    distributions = getattr(metadata, 'distribution', None)
    if distributions is not None:
        if not isinstance(distributions, list):
            distributions = [distributions]
        print("Available DataFileObject @id(s):")
        for d in distributions:
            _id = getattr(d, '@id', None)
            print(f"  - {_id}")
        print("\nYou may access tabular data from each DataFileObject via its '@id' as a record set.")
        available_record_sets = [_id for d in distributions for _id in [getattr(d, '@id', None)] if _id is not None]
    else:
        available_record_sets = []
        print("No distributions or record sets found in this Croissant package.")
else:
    print("Available record_set @id(s):")
    for rset in record_sets:
        print(f"  - {getattr(rset, '@id', None)}")
    available_record_sets = [getattr(rset, '@id', None) for rset in record_sets if getattr(rset, '@id', None) is not None]

# For demonstration, examine the fields in the first available record set (if present)
if available_record_sets:
    preview_records = dataset.records(record_set=available_record_sets[0])
    preview_rows = list(preview_records)
    if preview_rows:
        print(f"\nFields in the first record set (@id='{available_record_sets[0]}'):")
        print(list(preview_rows[0].keys()))
    else:
        print("No records found for the first record set.")
else:
    print("No usable record sets or distributions found. Please check the Croissant metadata.")

## 3. Data Extraction
Load data from each available record set (by `@id`) into DataFrames for further analysis.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Extract all available DataFileObject/record set @id's identified previously
dataframes = {}

for record_set_id in available_record_sets:
    print(f"\nLoading records from record set '@id': {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f" - Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print(" - No records found.")

# Preview the columns and first few rows of the first loaded DataFrame
if dataframes:
    first_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set '@id': {first_id}")
    print(dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())
else:
    print("No dataframes loaded. Please check if the record sets contain records.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing, such as filtering, normalization, and grouping by key fields, referencing columns by their `@id` as loaded above.

> _Note: For demonstration, we will select a numeric field (by column `@id`) that exists in the data for EDA._

In [ ]:
# Select one loaded DataFrame by its record set @id
if not dataframes:
    raise RuntimeError("No record set dataframes available for EDA.")

# Change these identifiers to match what is loaded in your environment
record_set_id = list(dataframes.keys())[0]  # Use first available
df = dataframes[record_set_id]

# Try to detect a likely numeric field, fallback to user select
numeric_field_id = None
for col in df.columns:
    try:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    except Exception:
        continue

if numeric_field_id is None:
    raise ValueError("No numeric column found for EDA. Please inspect df.columns and set a valid field id.")
print(f"Using numeric field id: {numeric_field_id}")

# Filtering: keep rows where the value is above a threshold (mean for example)
threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f}: {len(filtered_df)} rows")
display(filtered_df.head())

# Normalize
norm_field = f"{numeric_field_id}_normalized"
filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' added as column '{norm_field}':")
display(filtered_df[[numeric_field_id, norm_field]].head())

# Try to group by a non-numeric column
candidate_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
group_field_id = candidate_group_fields[0] if candidate_group_fields else None
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize numeric distributions and relationships using the selected field ids.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of a numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If we have grouping, show grouped mean as bar plot
if group_field_id:
    plt.figure(figsize=(10,4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=60)
    plt.title(f'{numeric_field_id} mean by {group_field_id}')
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:

- Load dataset metadata and record sets using their `@id` with `mlcroissant`.
- Explore available data and structure from a Croissant package schema.
- Extract, filter, normalize, and group data using field `@id`s.
- Visualize numeric distributions and groupings for rapid data understanding.

For further analysis, refer to the field and record set `@id`s exposed in the data overview steps. For new datasets, always inspect available fields using the notebook's overview section. 

See the [mlcroissant documentation](https://mlcommons.github.io/croissant) for advanced use cases, referencing, and schema extension.